In [4]:
import pandas as pd
import numpy as np

In [1]:
class ExcelSheetComparator:
    def __init__(self, file_path):
        self.file_path = file_path
        self.excel_file = pd.ExcelFile(file_path)
    def get_sheet_names(self):
        return self.excel_file.sheet_names
    def load_sheet(self, sheet_name):
        df = pd.read_excel(self.file_path, sheet_name=sheet_name)
        df.columns = df.columns.str.strip()
        return df
    def compare_sheets(self, old_sheet_name, new_sheet_name, 
                       key_column=None, firm_column=None, firm_filter=None):
        df_old = self.load_sheet(old_sheet_name)
        df_new = self.load_sheet(new_sheet_name)
        if firm_column and firm_filter:
            if firm_column in df_old.columns:
                df_old = df_old[df_old[firm_column] == firm_filter]
            if firm_column in df_new.columns:
                df_new = df_new[df_new[firm_column] == firm_filter]
        if key_column:
            old_keys = set(df_old[key_column].dropna())
            new_keys = set(df_new[key_column].dropna())
            new_entries = new_keys - old_keys
            missing_entries = old_keys - new_keys
            common_entries = old_keys & new_keys
            new_df = df_new[df_new[key_column].isin(new_entries)]
            missing_df = df_old[df_old[key_column].isin(missing_entries)]
            common_df = df_new[df_new[key_column].isin(common_entries)]
        else:
            old_tuples = set(df_old.itertuples(index=False, name=None))
            new_tuples = set(df_new.itertuples(index=False, name=None))
            new_tuples_only = new_tuples - old_tuples
            missing_tuples = old_tuples - new_tuples
            common_tuples = old_tuples & new_tuples
            new_df = pd.DataFrame(list(new_tuples_only), columns=df_new.columns)
            missing_df = pd.DataFrame(list(missing_tuples), columns=df_old.columns)
            common_df = pd.DataFrame(list(common_tuples), columns=df_new.columns)
        
        return {
            'new_entries': new_df,
            'missing_entries': missing_df,
            'common_entries': common_df,
            'summary': {
                'new_count': len(new_df),
                'missing_count': len(missing_df),
                'common_count': len(common_df),
                'old_total': len(df_old),
                'new_total': len(df_new)}}
    
    def print_comparison_summary(self, comparison_results):
        summary = comparison_results['summary']
        print("=" * 60)
        print("COMPARISON SUMMARY")
        print("=" * 60)
        print(f"Old sheet total entries: {summary['old_total']}")
        print(f"New sheet total entries: {summary['new_total']}")
        print(f"Common entries: {summary['common_count']}")
        print(f"New entries: {summary['new_count']}")
        print(f"Missing entries: {summary['missing_count']}")
        print("=" * 60)
    
    def get_unique_firms(self, sheet_name, firm_column):
        df = self.load_sheet(sheet_name)
        if firm_column in df.columns:
            return sorted(df[firm_column].dropna().unique().tolist())
        else:
            print(f"Warning: Column '{firm_column}' not found in sheet '{sheet_name}'")
            return []

In [5]:
path = r'C:\Users\ntaylor\Desktop\FullClientSearch.xlsx'
comparator = ExcelSheetComparator(path)

In [6]:
print("Available sheets:", comparator.get_sheet_names())

Available sheets: ['Names', '30Sep', '9Jun', '20May', '1May', '19Feb']


In [8]:
results = comparator.compare_sheets(
    old_sheet_name='9Jun',
    new_sheet_name='30Sep',
    key_column='First'  # Adjust this to match your actual column name
)

In [10]:
results['new_entries']

,First,Last,Firm,Position,Location
28,Shoney,Katz,Brevan Howard US Invst Mgmt LP,Portfolio Manager,New York
33,Tina,Tian,Brevan Howard US Invst Mgmt LP,Portfolio Manager,New York
93,Minal,Lavingia,Exoduspoint Capital Manage UK LLP,Portfolio Manager:Fixed Income,London
94,Jorn,Grodeland,Exoduspoint Capital Manage UK LLP,Portfolio Manager,London
117,Vyomakesh,Sridhar,Exoduspoint Capital Technologies Ltd,Portfolio Manager,London
152,Thibaut,Nocella,Verition Advisors UK Partners LLP,Portfolio Manager,London
237,Rajeev,Patel,Millennium Partners LP,Senior Portfolio Manager,New York
240,Rajeev,Patel,Millennium Partners LP,Senior Portfolio Manager,New York
266,Tobias,Weimann,Millennium Partners LP,Portfolio Manager,West Hollywood
599,Yeqi,Zhu,Verition Advisors UK Partners LLP,Portfolio Manager,London
